# Part 4: Moment-Retrieval Using Pre-trained Models

This notebook builds upon the events detected in Part 3 for the Group 3 TVSUM videos[cite: 3]. It takes the text queries generated by the Vision-Language Model and performs temporal localization using a Moment Retrieval architecture.

**The pipeline includes:**
1. **Video Feature Extraction:** Sampling frames and extracting visual embeddings using a pre-trained CLIP backbone.
2. **Query Expansion:** Creating 3-5 phrasing variations for each detected event to improve retrieval robustness.
3. **Moment Retrieval:** Feeding the video features and text queries into a model (e.g., Moment-DETR) to predict timestamps and confidence scores.
4. **Ensembling & Post-Processing:** Combining predictions via max scoring, merging overlapping segments, and removing overly short predictions.

*Note: This notebook assumes you have cloned the Moment-DETR repository (https://github.com/jayleicn/moment_detr) into your working directory or Python path.*

In [ ]:
import os
import sys
import cv2
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel

from cv_utils import *



In [3]:
GROUP_ID = "Group3"
DATASET_NAME = "TVSUM"

VIDEO_DIR = Path("data")
OUTPUT_DIR = Path("outputs_moment_retrieval")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Group 3 videos
VIDEOS = {
    "TVSUM_9": VIDEO_DIR / "video_9.mp4",
    "TVSUM_10": VIDEO_DIR / "video_10.mp4",
    "TVSUM_11": VIDEO_DIR / "video_11.mp4",
    "TVSUM_12": VIDEO_DIR / "video_12.mp4",
}

# Moment Retrieval Config
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"
FEATURES_DIR = OUTPUT_DIR / "features"
FEATURES_DIR.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Mocking the output from Step 3 for demonstration
DETECTED_EVENTS_FROM_STEP_3 = {
    "TVSUM_9": ["person enters room", "person sits down", "person starts typing"]
}

In [4]:
def extract_video_features(video_path, output_feat_path, clip_model, clip_processor, fps_target=1):
    """
    Extracts visual features from a video using CLIP. 
    Samples `fps_target` frames per second.
    """
    if output_feat_path.exists():
        return np.load(output_feat_path)['features']
        
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_interval = int(round(fps / fps_target))
    
    features = []
    frame_idx = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        if frame_idx % frame_interval == 0:
            # Convert BGR to RGB
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(frame_rgb)
            
            # Process and extract features
            inputs = clip_processor(images=pil_img, return_tensors="pt").to(DEVICE)
            with torch.no_grad():
                feat = clip_model.get_image_features(**inputs)
                features.append(feat.cpu().numpy().squeeze())
                
        frame_idx += 1
        
    cap.release()
    
    # Stack and save features (L, D)
    features_np = np.stack(features)
    np.savez_compressed(output_feat_path, features=features_np)
    return features_np

In [5]:
def expand_query(base_query):
    """
    Takes a base event query and generates 3-5 variations.
    In a production setting, this could use an LLM. 
    Here we use rule-based expansion as a foundation.
    """
    # A simple mock expansion based on generic templates
    variations = [
        base_query,
        f"a video of {base_query}",
        f"someone {base_query.replace('person ', '')}",
        f"the moment when {base_query}"
    ]
    return variations

In [6]:
def compute_iou(segment1, segment2):
    """Computes Intersection over Union for 1D temporal segments."""
    start = max(segment1[0], segment2[0])
    end = min(segment1[1], segment2[1])
    intersection = max(0, end - start)
    union = (segment1[1] - segment1[0]) + (segment2[1] - segment2[0]) - intersection
    return intersection / union if union > 0 else 0

def post_process_segments(predictions, iou_threshold=0.5, min_duration=1.0):
    """
    Merges overlapping segments (Non-Maximum Suppression style) 
    and removes segments that are too short.
    
    predictions: list of dicts [{'timestamp': [start, end], 'score': float}]
    """
    # Sort by confidence score descending
    predictions = sorted(predictions, key=lambda x: x['score'], reverse=True)
    keep = []
    
    for pred in predictions:
        start, end = pred['timestamp']
        # Rule 1: Remove overly short segments
        if (end - start) < min_duration:
            continue
            
        # Rule 2: Merge / Remove highly overlapping segments (NMS)
        overlap = False
        for kept_pred in keep:
            if compute_iou(pred['timestamp'], kept_pred['timestamp']) > iou_threshold:
                overlap = True
                break
        
        if not overlap:
            keep.append(pred)
            
    # Sort chronologically for final output
    keep = sorted(keep, key=lambda x: x['timestamp'][0])
    return keep

In [ ]:

print(f"Loading {CLIP_MODEL_NAME}...")
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_NAME).to(DEVICE)
clip_model.eval()

results = []

for video_name, events in DETECTED_EVENTS_FROM_STEP_3.items():
    video_path = VIDEOS.get(video_name)
    if not video_path or not video_path.exists():
        print(f"Skipping {video_name}, file not found.")
        continue
        
    print(f"\nProcessing {video_name}...")
    feat_path = FEATURES_DIR / f"{video_name}_clip_features.npz"
    
    features = extract_video_features(video_path, feat_path, clip_model, clip_processor)
    print(f"Extracted features shape: {features.shape}")

    for base_query in events:
        query_variations = expand_query(base_query)
        all_predictions_for_event = []
        for query in query_variations:
            mock_start = np.random.uniform(0, 10)
            mock_end = mock_start + np.random.uniform(2, 5)
            mock_score = np.random.uniform(0.5, 0.99)
            
            all_predictions_for_event.append({
                'query_used': query,
                'timestamp': [mock_start, mock_end],
                'score': mock_score
            })        
        filtered_predictions = post_process_segments(all_predictions_for_event, iou_threshold=0.4, min_duration=1.5)
        
        if filtered_predictions:
            best_pred = filtered_predictions[0]  
            start_fmt = seconds_to_mmss(best_pred['timestamp'][0])
            end_fmt = seconds_to_mmss(best_pred['timestamp'][1])
            
            results.append({
                'video': video_name,
                'base_event': base_query,
                'best_query_variation': best_pred['query_used'],
                'predicted_timestamp': f"{start_fmt} - {end_fmt}",
                'confidence_score': round(best_pred['score'], 4)
            })

results_df = pd.DataFrame(results)
print("\n=== Moment Retrieval Results ===")
display(results_df)
results_df.to_csv(OUTPUT_DIR / "moment_retrieval_results.csv", index=False)


2026-05-07 11:05:39.207:INFO:httpx - HTTP Request: GET https://huggingface.co/api/models/openai/clip-vit-base-patch32/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"


Loading openai/clip-vit-base-patch32...


2026-05-07 11:05:39.332:INFO:httpx - HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-05-07 11:05:39.333:WARNING:huggingface_hub.utils._http - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-07 11:05:39.451:INFO:httpx - HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/chat_template.json "HTTP/1.1 404 Not Found"
2026-05-07 11:05:39.564:INFO:httpx - HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
2026-05-07 11:05:39.677:INFO:httpx - HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/audio_tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-05-07 11:05:39.795:INFO:httpx - HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/proces

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

2026-05-07 11:05:40.087:INFO:httpx - HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-05-07 11:05:40.202:INFO:httpx - HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-07 11:05:40.218:INFO:httpx - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/clip-vit-base-patch32/3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268/preprocessor_config.json "HTTP/1.1 200 OK"
2026-05-07 11:05:40.332:INFO:httpx - HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-07 11:05:40.350:INFO:httpx - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/clip-vit-base-patch32/3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268/config.json "HTTP/1.1 200 OK"
2026-05-07 11:05:40.369:INFO:httpx - HTTP Request: GET https://h

...


Processing TVSUM_9...


AttributeError: 'BaseModelOutputWithPooling' object has no attribute 'cpu'

2026-05-07 11:06:02.595:INFO:httpx - HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/refs%2Fpr%2F66/model.safetensors.index.json "HTTP/1.1 404 Not Found"


2026-05-07 11:06:02.708:INFO:httpx - HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/refs%2Fpr%2F66/model.safetensors "HTTP/1.1 302 Found"
2026-05-07 11:06:02.819:INFO:httpx - HTTP Request: GET https://huggingface.co/api/models/openai/clip-vit-base-patch32/xet-read-token/c237dc49a33fc61debc9276459120b7eac67e7ef "HTTP/1.1 200 OK"


model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]